**Objetivo:** construir (1) o **Star Schema** para o BI, (2) a tabela de contexto **`gold_genai_movies_context`** para o Vector Search
do time de IA e (3) responder às **perguntas de negócio** com `display()`.

**Decisões de modelagem**
- **Chaves substitutas** via `row_number()` ordenado pela chave natural: determinístico (mesma entrada → mesmas chaves) e denso.
- Cada dimensão é gravada e **relida da Gold** antes de alimentar fato/pontes, garantindo que todas usem exatamente as mesmas SKs.
- Fato com **1 linha por filme**: parte da `dim_movies` e usa `LEFT JOIN` com tabelas já deduplicadas por `id_filme` (não multiplica linhas).
- Relações N:N (gênero, pessoa, produtora) ficam em **tabelas-ponte**, nunca na fato.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "workspace", "Catálogo (Unity Catalog)")
catalog = dbutils.widgets.get("catalog").strip()

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Decisão de negócio: a fato traz TODOS os filmes da dim_movies (1:1 com a dimensão), preservando o grão "1 linha por filme".
# Se preferir restringir a fato a filmes com status 'Lançado', mude para True.
SOMENTE_LANCADOS_NA_FATO = False


def write_gold(df, tabela, schema):
    """Aplica o schema (tipos/ordem do escopo) e grava em Delta (overwrite idempotente)."""
    df = df.select(*[F.col(c).cast(t).alias(c) for c, t in schema.items()])
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"[ok] {tabela}: {spark.table(tabela).count():,} linhas")


def blank_to_null(c):
    """trim + string vazia -> NULL (para o fallback funcionar também com strings vazias)."""
    t = F.trim(c.cast("string"))
    return F.when(t == "", F.lit(None)).otherwise(t)


PESSOAS = ["Ator", "Diretor", "Roteirista"]

In [0]:
# --- dim_movies ---------------------------------------------------------------------------------
s_info = spark.table("silver.tb_info_filmes")
w_filme = Window.orderBy("id_filme")

dim_movies = s_info.select(
    F.row_number().over(w_filme).alias("sk_movie_id"),
    "id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos",
    "idioma_original", "status_filme", "sinopse",
)
write_gold(dim_movies, "gold.dim_movies", {
    "sk_movie_id": "bigint", "id_filme": "string", "titulo": "string", "data_lancamento": "date",
    "ano_lancamento": "int", "duracao_minutos": "int", "idioma_original": "string",
    "status_filme": "string", "sinopse": "string",
})

# --- dim_genres ---------------------------------------------------------------------------------
s_generos = spark.table("silver.tb_generos")
dim_genres = (
    s_generos.select(F.col("genero").alias("nome_genero")).distinct()
    .select(F.row_number().over(Window.orderBy("nome_genero")).alias("sk_genre_id"), "nome_genero")
)
write_gold(dim_genres, "gold.dim_genres", {"sk_genre_id": "bigint", "nome_genero": "string"})

# --- dim_people (Ator, Diretor, Roteirista) e dim_companies (Produtora) -------------------------
s_pe = spark.table("silver.tb_pessoas_empresas")

dim_people = (
    s_pe.filter(F.col("tipo_entidade").isin(PESSOAS))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct()
    .select(
        F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).alias("sk_person_id"),
        "nome_pessoa", "tipo_pessoa",
    )
)
write_gold(dim_people, "gold.dim_people", {"sk_person_id": "bigint", "nome_pessoa": "string", "tipo_pessoa": "string"})

dim_companies = (
    s_pe.filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora")).distinct()
    .select(F.row_number().over(Window.orderBy("nome_produtora")).alias("sk_company_id"), "nome_produtora")
)
write_gold(dim_companies, "gold.dim_companies", {"sk_company_id": "bigint", "nome_produtora": "string"})

# Relê as dimensões já gravadas: a partir daqui todo mundo usa as mesmas SKs
dim_movies = spark.table("gold.dim_movies")
dim_genres = spark.table("gold.dim_genres")
dim_people = spark.table("gold.dim_people")
dim_companies = spark.table("gold.dim_companies")
chaves_filme = dim_movies.select("sk_movie_id", "id_filme")

# --- dim_reviews: avaliações resumidas por filme ------------------------------------------------
resumo_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .join(chaves_filme, "id_filme", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.count(F.lit(1)).alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),   # média ignora notas NULL
    )
)
dim_reviews = resumo_reviews.select(
    F.row_number().over(Window.orderBy("sk_movie_id")).alias("sk_review_id"),
    "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios",
)
write_gold(dim_reviews, "gold.dim_reviews", {
    "sk_review_id": "bigint", "sk_movie_id": "bigint",
    "qtd_avaliacoes_usuarios": "int", "nota_media_usuarios": "double",
})

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[ok] gold.dim_movies: 97,879 linhas
[ok] gold.dim_genres: 20 linhas
[ok] gold.dim_people: 420,180 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[ok] gold.dim_companies: 45,899 linhas
[ok] gold.dim_reviews: 27,303 linhas


Tabelas-ponte (N:N):
Junção pela chave natural do filme e pelo nome da entidade; selecionamos só as duas SKs e removemos repetições.


In [0]:
bridge_genre = (
    s_generos.join(chaves_filme, "id_filme", "inner")
    .join(dim_genres, s_generos["genero"] == dim_genres["nome_genero"], "inner")
    .select("sk_movie_id", "sk_genre_id").distinct()
)
write_gold(bridge_genre, "gold.bridge_movie_genre", {"sk_movie_id": "bigint", "sk_genre_id": "bigint"})

pessoas_filme = (
    s_pe.filter(F.col("tipo_entidade").isin(PESSOAS))
    .select("id_filme", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)
bridge_person = (
    pessoas_filme.join(chaves_filme, "id_filme", "inner")
    .join(dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id").distinct()
)
write_gold(bridge_person, "gold.bridge_movie_person", {"sk_movie_id": "bigint", "sk_person_id": "bigint"})

produtoras_filme = (
    s_pe.filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
)
bridge_company = (
    produtoras_filme.join(chaves_filme, "id_filme", "inner")
    .join(dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id").distinct()
)
write_gold(bridge_company, "gold.bridge_movie_company", {"sk_movie_id": "bigint", "sk_company_id": "bigint"})

[ok] gold.bridge_movie_genre: 140,822 linhas
[ok] gold.bridge_movie_person: 763,305 linhas
[ok] gold.bridge_movie_company: 117,357 linhas


Grão: **1 linha por filme**. `LEFT JOIN` com financeiro e métricas (ambos já únicos por `id_filme` na Silver) ⇒ os joins não duplicam linhas.

In [0]:
s_fin = spark.table("silver.tb_financeiro_filmes")
s_met = spark.table("silver.tb_metricas_engajamento")

base_fato = dim_movies.select("sk_movie_id", "id_filme", "status_filme")
if SOMENTE_LANCADOS_NA_FATO:
    base_fato = base_fato.filter(F.col("status_filme") == "Lançado")

fact = (
    base_fato
    .join(s_fin, "id_filme", "left")
    .join(s_met, "id_filme", "left")
)
write_gold(fact, "gold.fact_movies_performance", {
    "sk_movie_id": "bigint",
    "orcamento_usd": "decimal(18,2)", "receita_usd": "decimal(18,2)", "lucro_usd": "decimal(18,2)",
    "orcamento_brl": "decimal(18,2)", "receita_brl": "decimal(18,2)", "lucro_brl": "decimal(18,2)",
    "popularidade": "double",
    "nota_media_tmdb": "double", "qtd_votos_tmdb": "int",
    "nota_media_imdb": "double", "qtd_votos_imdb": "int",
})

# Validação de grão: nenhuma duplicação por causa dos joins
f = spark.table("gold.fact_movies_performance")
assert f.count() == f.select("sk_movie_id").distinct().count(), "Fato com sk_movie_id duplicado!"
print("Grão da fato preservado (1 linha por filme): OK")

[ok] gold.fact_movies_performance: 97,879 linhas
Grão da fato preservado (1 linha por filme): OK


`concat()` devolve **NULL** se qualquer parte for NULL — um único diretor ou sinopse ausente faria o filme sumir do contexto.
Solução: `coalesce(campo, fallback)` **em cada campo, antes** do `concat` (e `blank_to_null` para tratar string vazia como ausente).

In [0]:
fato = spark.table("gold.fact_movies_performance")

# Atores principais: pelas pontes, ordenados pela posição no crédito original (ordem_credito da Silver), top 5
ordem_atores = (
    s_pe.filter(F.col("tipo_entidade") == "Ator")
    .select("id_filme", F.col("nome_entidade").alias("nome_pessoa"), "ordem_credito")
)
atores = (
    spark.table("gold.bridge_movie_person")
    .join(dim_people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .join(chaves_filme, "sk_movie_id")
    .join(ordem_atores, ["id_filme", "nome_pessoa"], "left")
    .groupBy("sk_movie_id")
    .agg(
        F.sort_array(
            F.collect_list(F.struct(F.coalesce(F.col("ordem_credito"), F.lit(999999)).alias("ordem"), F.col("nome_pessoa")))
        ).alias("lista")
    )
    .select(
        "sk_movie_id",
        F.array_join(F.slice(F.transform("lista", lambda x: x["nome_pessoa"]), 1, 5), ", ").alias("atores_principais"),
    )
)

# Diretores (pode haver mais de um): agregados em uma única string
diretores = (
    spark.table("gold.bridge_movie_person")
    .join(dim_people.filter(F.col("tipo_pessoa") == "Diretor"), "sk_person_id")
    .groupBy("sk_movie_id")
    .agg(F.array_join(F.sort_array(F.collect_set("nome_pessoa")), ", ").alias("diretores"))
)

base_ctx = (
    dim_movies
    .join(fato.select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
    .join(atores, "sk_movie_id", "left")
    .join(diretores, "sk_movie_id", "left")
)

# Cada trecho já recebe seu fallback: concat() nunca enxerga um NULL
titulo_txt = F.coalesce(blank_to_null(F.col("titulo")), F.lit("sem título"))
ano_txt = F.coalesce(F.concat(F.lit("no ano de "), F.col("ano_lancamento").cast("string")), F.lit("em data não informada"))
receita_txt = F.coalesce(F.concat(F.lit("US$ "), F.format_number("receita_usd", 2)), F.lit("um valor não divulgado"))
orcamento_txt = F.coalesce(F.concat(F.lit("US$ "), F.format_number("orcamento_usd", 2)), F.lit("um valor não divulgado"))
atores_txt = F.coalesce(blank_to_null(F.col("atores_principais")), F.lit("elenco não informado"))
diretor_txt = F.coalesce(blank_to_null(F.col("diretores")), F.lit("diretor não informado"))
# remove ponto/espaço finais da sinopse para não gerar ".." ao fechar a frase
sinopse_txt = F.coalesce(
    blank_to_null(F.regexp_replace(F.col("sinopse"), r"[\.\s]+$", "")),
    F.lit("sinopse não disponível"),
)

llm_doc = F.concat(
    F.lit("O filme "), titulo_txt,
    F.lit(", lançado "), ano_txt,
    F.lit(", faturou "), receita_txt,
    F.lit(" e teve um custo de "), orcamento_txt,
    F.lit(". Estrelado por "), atores_txt,
    F.lit(" e dirigido por "), diretor_txt,
    F.lit(", o filme possui a seguinte sinopse: "), sinopse_txt,
    F.lit("."),
)

contexto = base_ctx.select(
    F.col("id_filme").alias("movie_id"),
    titulo_txt.alias("title"),
    llm_doc.alias("llm_context_document"),
)
write_gold(contexto, "gold.gold_genai_movies_context", {
    "movie_id": "string", "title": "string", "llm_context_document": "string",
})

# O Vector Search (Delta Sync Index) exige Change Data Feed habilitado na tabela de origem
spark.sql("ALTER TABLE gold.gold_genai_movies_context SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Validações: nenhum filme pode sumir por causa de NULL na concatenação
ctx = spark.table("gold.gold_genai_movies_context")
assert ctx.count() == spark.table("gold.dim_movies").count(), "Há filmes faltando na tabela de contexto!"
assert ctx.filter(F.col("llm_context_document").isNull()).count() == 0, "Há documentos NULL!"
print("Contexto de IA: 1 documento por filme, nenhum NULL: OK")
display(ctx.limit(5))

[ok] gold.gold_genai_movies_context: 97,879 linhas
Contexto de IA: 1 documento por filme, nenhum NULL: OK


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou um valor não divulgado e teve um custo de um valor não divulgado. Estrelado por Izzy Jones, Erika Alexander, Aron Von Andrian, Steven Michael-o’hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou um valor não divulgado e teve um custo de um valor não divulgado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou um valor não divulgado e teve um custo de um valor não divulgado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou um valor não divulgado e teve um custo de um valor não divulgado. Estrelado por Marcello Urgeghe, João Pedro Bénard, Isabel Abreu, Inês Pronto e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou um valor não divulgado e teve um custo de um valor não divulgado. Estrelado por Soulayman Rkiba, Gabrielle Cohen, Claire Chust, Maxime Pambet, Biyouna e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: sinopse não disponível."


In [0]:
# PKs únicas em todas as dimensões
for tabela, pk in [
    ("gold.dim_movies", "sk_movie_id"), ("gold.dim_genres", "sk_genre_id"), ("gold.dim_people", "sk_person_id"),
    ("gold.dim_companies", "sk_company_id"), ("gold.dim_reviews", "sk_review_id"),
]:
    d = spark.table(tabela)
    assert d.count() == d.select(pk).distinct().count(), f"{tabela}: {pk} duplicada"
print("PKs únicas: OK")

# Pontes sem chaves órfãs
for ponte, fk, dim, pk in [
    ("gold.bridge_movie_genre", "sk_genre_id", "gold.dim_genres", "sk_genre_id"),
    ("gold.bridge_movie_person", "sk_person_id", "gold.dim_people", "sk_person_id"),
    ("gold.bridge_movie_company", "sk_company_id", "gold.dim_companies", "sk_company_id"),
    ("gold.bridge_movie_genre", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_person", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
    ("gold.bridge_movie_company", "sk_movie_id", "gold.dim_movies", "sk_movie_id"),
]:
    df_ponte, df_dim = spark.table(ponte), spark.table(dim)
    orfas = df_ponte.join(df_dim, df_ponte[fk] == df_dim[pk], "left_anti").count()
    assert orfas == 0, f"{ponte}.{fk} tem {orfas} chaves órfãs"
print("Pontes sem chaves órfãs: OK")

PKs únicas: OK
Pontes sem chaves órfãs: OK
